# 🧠 BraTS 2D JEPA Representation Learning & Segmentation Benchmark
### Complete Kaggle GPU Training, Fine-Tuning, Probing & Evaluation Runner

This standalone Kaggle notebook provides a complete execution pipeline for evaluating Self-Supervised Learning (SSL) architectures on BraTS 2D multi-modal MRI scans.

**Architectures evaluated:**
- **I-JEPA** (Image-based Joint-Embedding Predictive Architecture)
- **SigReg JEPA** (Sinkhorn Information-Theoretic Regularization)
- **VisReg JEPA** (Variance-Invariance-Covariance Regularization)
- **BraTS 2D UNet** (Classical Supervised Baseline)
- **BraTS 2D nnU-Net** (Supervised SOTA with Deep Supervision)

--- 
### ⚡ Hardware & Acceleration Recommendations
- **Accelerator:** GPU T4 x 1 or GPU T4 x 2 (Tesla T4, 16GB VRAM each)
- **Mixed Precision (AMP):** Enabled by default via `--amp` (`torch.amp.autocast('cuda')` + `GradScaler`) for ~3x throughput speedup and 50% lower VRAM usage.
- **Internet Access:** Turn **ON** (in right sidebar: Settings -> Internet -> Turn On) to install dependencies.
- **Persistence:** All checkpoints, logs, and benchmark summaries are saved to `/kaggle/working/outputs/` and compressed to `outputs.zip` for 1-click download.

## 1. Hardware & CUDA Environment Verification
Verify GPU allocation and CUDA driver state.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
    print(f"Memory Reserved:  {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Please go to Notebook Settings -> Accelerator -> GPU T4!")

## 2. Dependencies Installation
Install `monai` (Medical Open Network for AI) for medical image metric computation (Hausdorff Distance HD95, Dice).

In [ ]:
!pip install -q --no-cache-dir monai
import monai
print(f"MONAI installed successfully: v{monai.__version__}")

## 3. Codebase Extraction & Setup
Find and unpack `thesis_2d_code.zip` from `/kaggle/input/` and install the package in editable mode.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/thesis_2d")
work_dir.mkdir(parents=True, exist_ok=True)

# Search for thesis_2d_code.zip across /kaggle/input
code_zips = list(Path("/kaggle/input").glob("**/thesis_2d_code.zip"))
if code_zips:
    src_zip = code_zips[0]
    print(f"Found codebase zip: {src_zip}")
    with zipfile.ZipFile(src_zip, 'r') as zf:
        zf.extractall(work_dir)
    print(f"Successfully extracted codebase to: {work_dir}")
else:
    print(f"Note: thesis_2d_code.zip not found in /kaggle/input.")
    if (work_dir / "src").exists():
        print(f"Existing codebase found in {work_dir}.")
    else:
        print("Please ensure your uploaded code dataset is attached to this notebook!")

# Install codebase in editable mode
!pip install -q -e /kaggle/working/thesis_2d

# Change working directory to codebase root
os.chdir(work_dir)
sys.path.insert(0, str(work_dir / "src"))
print(f"Current working directory: {os.getcwd()}")

## 4. Dataset Discovery & Health Checks
Verify the dataset mount points and slice integrity.

In [ ]:
import pandas as pd
from brats_jepa.config import get_dataset_dir, get_metadata_path
from brats_jepa.data import BraTS2DDataset

print("=== DATASET DISCOVERY ===")
gli_meta = get_metadata_path("brats_gli_2d")
print(f"BraTS-GLI-2D Metadata Path: {gli_meta} (Exists: {gli_meta.exists()})")

if gli_meta.exists():
    df_gli = pd.read_csv(gli_meta)
    print(f"  Total Slices: {len(df_gli):,}")
    print(f"  Split Distribution:\n{df_gli['split'].value_counts().to_string()}")
    
    # Quick dataset tensor check
    ds = BraTS2DDataset(metadata_csv=gli_meta, split="train")
    sample = ds[0]
    print(f"  Sample Image Tensor: shape={sample['image'].shape}, dtype={sample['image'].dtype}")
    print(f"  Sample Label Tensor: shape={sample['label'].shape}, dtype={sample['label'].dtype}")
    print("  ✓ Dataset check passed!")
else:
    print("  ⚠️ Metadata not found! Available files in /kaggle/input:")
    for p in Path("/kaggle/input").glob("*"):
        print(f"    - {p}")

men_meta = get_metadata_path("brats_men_rt_2d")
print(f"\nBraTS-MEN-RT (OOD) Metadata: {men_meta} (Exists: {men_meta.exists()})")

--- 
## 5. Phase 1: Self-Supervised JEPA Pre-training
Pre-train the Vision Transformer encoders on 2D slices without any segmentation labels.
Runs with Mixed Precision (`--amp`), default 50 epochs, batch size 16 (or 8).

In [ ]:
# Pre-train Standard I-JEPA (Smooth L1 Target Prediction)
!python scripts/train_jepa.py --model_type ijepa --epochs 50 --batch_size 16 --amp

In [ ]:
# Pre-train SigReg JEPA (Sinkhorn Distance Regularization)
!python scripts/train_jepa.py --model_type sigreg_jepa --epochs 50 --batch_size 16 --amp

In [ ]:
# Pre-train VisReg JEPA (Variance-Invariance-Covariance Regularization)
!python scripts/train_jepa.py --model_type visreg_jepa --epochs 50 --batch_size 16 --amp

--- 
## 6. Phase 2: Supervised Baselines (UNet & nnU-Net)
Train fully-supervised reference models on 100% labeled training slices for 30 epochs with `--amp`.

In [ ]:
# Train BraTS 2D UNet Baseline
!python scripts/train_unet.py --epochs 30 --batch_size 16 --amp

In [ ]:
# Train BraTS 2D nnU-Net (SOTA with Deep Supervision)
!python scripts/train_nnunet.py --epochs 30 --batch_size 16 --amp

--- 
## 7. Phase 3: Downstream Segmentation Fine-Tuning
Fine-tune the pre-trained JEPA encoders with the convolutional segmentation decoder head on 100% labels for 30 epochs.

In [ ]:
# Fine-tune I-JEPA Downstream Segmentation
!python scripts/train_downstream.py --model_type ijepa --epochs 30 --batch_size 16 --amp

In [ ]:
# Fine-tune SigReg JEPA Downstream Segmentation
!python scripts/train_downstream.py --model_type sigreg_jepa --epochs 30 --batch_size 16 --amp

In [ ]:
# Fine-tune VisReg JEPA Downstream Segmentation
!python scripts/train_downstream.py --model_type visreg_jepa --epochs 30 --batch_size 16 --amp

--- 
## 8. Phase 4: Downstream Evaluation & Representation Probing
Evaluate all models on the held-out test split, computing:
- Dice Similarity Coefficient (DSC)
- Intersection over Union (IoU)
- 95th Percentile Hausdorff Distance (HD95 in pixels)
- Linear Probing Accuracy
- Effective Representation Rank (singular value spectrum entropy)
- Average Pairwise Cosine Similarity (representation collapse metric)

In [ ]:
!python scripts/evaluate.py

--- 
## 9. Phase 5: Low-Data Label Efficiency Benchmark (1% to 100% Labels)
Systematic comparison of label efficiency across 1%, 5%, 10%, 25%, 50%, and 100% annotations with Random Modality Dropout (p=0.25).

In [ ]:
!python scripts/evaluate_low_data.py --epochs 30 --batch_size 16 --amp --exp_version kaggle_low_data

--- 
## 10. Phase 6: Out-of-Distribution (OOD) Scanner & Cross-Pathology Benchmarks
Evaluate robustness against simulated scanner domain shifts (Gaussian noise, blur, intensity bias) and zero-shot transfer to BraTS-MEN-RT (Meningioma radiotherapy contours).

In [ ]:
# 1. Scanner Domain Shifts (In-distribution Glioma with Scanner Perturbations)
!python scripts/evaluate_ood.py --exp_version kaggle_ood

# 2. Cross-Pathology Meningioma Zero-Shot Transfer
!python scripts/evaluate_men_rt_ood.py --max_samples 5000 --exp_version kaggle_men_rt_ood

--- 
## 11. Phase 7: Figure Generation & Inline Display
Generate publication-grade comparison plots and display them directly in this notebook.

In [ ]:
!python scripts/generate_figures.py

from IPython.display import Image, display
from pathlib import Path

figures_dir = Path("/kaggle/working/outputs/figures")
if not figures_dir.exists():
    figures_dir = Path("outputs/figures")

for fig_path in sorted(figures_dir.glob("*.png")):
    print(f"\nFigure: {fig_path.name}")
    display(Image(filename=str(fig_path), width=750))

--- 
## 12. Phase 8: Package Outputs for Download
Compress the entire `outputs/` directory (checkpoints, metrics CSVs, figures, logs) into `/kaggle/working/outputs.zip`.
You can download `outputs.zip` directly from the Kaggle Output viewer on the right panel!

In [ ]:
!zip -q -r /kaggle/working/outputs.zip /kaggle/working/outputs
import os
size_mb = os.path.getsize("/kaggle/working/outputs.zip") / (1024 * 1024)
print(f"✓ outputs.zip created successfully! Total Size: {size_mb:.2f} MB")
print("You can now download outputs.zip from the right-hand Output panel in Kaggle!")